# TT decomposition and superweights in LLMs


In [ ]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=2

In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'src').exists() else CWD.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo root:', REPO_ROOT)
print('Has src:', (REPO_ROOT / 'src').exists())

In [ ]:
import gc
import json
import math
from collections import defaultdict

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.tt_llm import (
    TTLinear,
    candidates_table,
    capture_layerwise_maxima,
    cleanup_memory,
    compare_entry_with_dense,
    entry_trace_table,
    find_superweight_candidates,
    find_superweights_iterative,
    format_cuda_memory,
    get_module_by_name,
    infer_input_device,
    replace_llama_ffn_with_tt,
    replace_tt_with_dense_reconstruction,
    sample_entry_error_summary,
    tt_entry_contribution_maps,
)

try:
    from src.utils.eval import eval_ppl
except Exception:
    eval_ppl = None

pd.set_option('display.max_colwidth', 200)
torch.set_grad_enabled(False)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Configuration

In [ ]:
MODEL_NAME = 'meta-llama/Llama-2-7b-hf'
USE_KNOWN_SUPERWEIGHTS = True

KNOWN_SUPERWEIGHTS = {
    'meta-llama/Llama-2-7b-hf': [('mlp.down_proj', 1, 2533, 7890)],
    'mistralai/Mistral-7B-v0.1': [('mlp.down_proj', 1, 2070, 7310)],
    'allenai/OLMo-1B-0724-hf': [('mlp.down_proj', 1, 1764, 1710), ('mlp.down_proj', 2, 1764, 8041)],
    'allenai/OLMo-7B-0724-hf': [('mlp.down_proj', 1, 269, 7467), ('mlp.down_proj', 2, 269, 8275), ('mlp.down_proj', 7, 269, 453), ('mlp.down_proj', 24, 269, 2300)],
}

DISCOVERY_PROMPT = 'Apple Inc. is a worldwide tech company.'
GENERATION_PROMPT = 'The theory of tensor train decomposition for neural networks suggests that'
DISCOVERY_THRESHOLD = 50.0
REQUIRE_SAME_TOKEN = True
MAX_SUPERWEIGHTS = 6

TT_RANKS = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 2000, 3000, 4000, 5000]
ORDER = 12
PROJECTIONS = ('down_proj',)
TOKEN_CHUNK_SIZE = 128
DECOMPOSE_DEVICE = 'cpu'
DECOMPOSE_DTYPE = torch.float64

RUN_PPL = False
DATASETS = ['wikitext2']
SEQLEN = 1024

PLOT_TOPK_OUTLIERS = 64
PLOT_RANDOM_SAMPLES = 256
ANALYSIS_PROMPT = DISCOVERY_PROMPT
OUTPUT_JSON = REPO_ROOT / 'tt_superweight_results.json'

In [ ]:
def clean_memory(*objs):
    for obj in objs:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_model_and_tokenizer(model_name: str, dtype=torch.float16, device_map='auto', low_cpu_mem_usage=True):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False, trust_remote_code=True)
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map=device_map,
        low_cpu_mem_usage=low_cpu_mem_usage,
        trust_remote_code=True,
    )
    model.eval()
    if hasattr(model.config, 'use_cache'):
        model.config.use_cache = False
    return model, tokenizer


@torch.no_grad()
def generate_text(model, tokenizer, prompt, max_new_tokens=48):
    encoded = tokenizer(prompt, return_tensors='pt')
    encoded = {k: v.to(infer_input_device(model)) for k, v in encoded.items()}
    out = model.generate(
        **encoded,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def known_superweight_rows(model_name: str):
    rows = []
    for module_name, layer_idx, row, col in KNOWN_SUPERWEIGHTS.get(model_name, []):
        rows.append({'module_name': module_name, 'layer_idx': layer_idx, 'row': row, 'col': col})
    return pd.DataFrame(rows)


def weight_coordinate_to_module_name(module_name: str, layer_idx: int) -> str:
    return f'model.layers.{layer_idx}.{module_name}'


def dense_weight_rank_stats(weight: torch.Tensor, coords):
    flat_abs = weight.detach().abs().reshape(-1).float().cpu()
    rows = []
    for row, col in coords:
        value = float(weight[row, col].float().cpu().item())
        abs_value = abs(value)
        strict_larger = int((flat_abs > abs_value).sum().item())
        rank = strict_larger + 1
        percentile = 100.0 * (1.0 - strict_larger / max(len(flat_abs), 1))
        rows.append({
            'row': row,
            'col': col,
            'value': value,
            'abs_value': abs_value,
            'abs_rank': rank,
            'top_percentile': percentile,
        })
    return pd.DataFrame(rows)


def plot_layerwise_maxima(input_df: pd.DataFrame, output_df: pd.DataFrame, title: str):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, df, name in zip(axes, [input_df, output_df], ['input', 'output']):
        ax.plot(df['layer_idx'], df['value'], marker='o')
        spikes = df[df['abs_value'] >= DISCOVERY_THRESHOLD]
        if len(spikes) > 0:
            ax.scatter(spikes['layer_idx'], spikes['value'], s=80, marker='s')
        ax.set_xlabel('Layer')
        ax.set_ylabel('Max activation')
        ax.set_title(f'{name} maxima')
        ax.grid(True)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_error_scatter(df: pd.DataFrame, title: str, use_log=True):
    plt.figure(figsize=(6.5, 4.5))
    markers = {'superweight': 's', 'top_outlier': 'o', 'random': '.'}
    for tag, group in df.groupby('tag'):
        plt.scatter(group['original_abs'], group['abs_error'], label=tag, marker=markers.get(tag, 'o'), alpha=0.8)
    if use_log:
        plt.xscale('log')
        plt.yscale('log')
    plt.ylim(0, 1)
    plt.xlabel('|original weight|')
    plt.ylabel('|TT error|')
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()


def plot_core_contribution_maps(
    tt_module: TTLinear,
    row: int,
    col: int,
    *,
    title: str = 'title',
    figsize_per_core: float = 4.2,
    panel_scale: float = 1.15,
    cmap_name: str = 'magma',
    zero_color: str = 'black',
    zero_tol: float = 0.0,
    normalize: str = 'global',
    use_abs: bool = True,
    ncols: int = 4,
    wspace: float = 0.18,
    hspace: float = 0.28,
    show_axis_labels: bool = True,
):
    import numpy as np
    import matplotlib.pyplot as plt

    maps = tt_entry_contribution_maps(tt_module, row=row, col=col)
    if use_abs:
        maps = [m.detach().abs().float().cpu().numpy() for m in maps]
    else:
        maps = [m.detach().float().cpu().numpy() for m in maps]

    if len(maps) == 0:
        raise ValueError("No contribution maps found.")

    if normalize not in {'global', 'per_core'}:
        raise ValueError("normalize must be 'global' or 'per_core'")

    if normalize == 'global':
        global_max = max(float(m.max()) for m in maps)
        global_max = max(global_max, 1e-12)
        plot_maps = [m / global_max for m in maps]
    else:
        plot_maps = []
        for m in maps:
            local_max = max(float(m.max()), 1e-12)
            plot_maps.append(m / local_max)

    n = len(plot_maps)
    ncols = min(ncols, n)
    nrows = math.ceil(n / ncols)

    fig_w = figsize_per_core * ncols * panel_scale
    fig_h = figsize_per_core * nrows * panel_scale

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(fig_w, fig_h),
        constrained_layout=True,
    )
    axes = np.array(axes).reshape(-1)

    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad(color=zero_color)

    for ax in axes[n:]:
        ax.axis('off')

    im = None
    for idx, (ax, m) in enumerate(zip(axes, plot_maps)):
        masked = np.ma.masked_where(np.abs(m) <= zero_tol, m)

        im = ax.imshow(
            masked,
            aspect='auto',
            vmin=0.0,
            vmax=1.0,
            cmap=cmap,
            interpolation='nearest',
        )

        ax.set_title(f'core {idx}', fontsize=12)

        row_id = idx // ncols
        col_id = idx % ncols

        if show_axis_labels and row_id == nrows - 1:
            ax.set_xlabel('right rank index')
        else:
            ax.set_xlabel('')

        # only show y labels on first column
        if show_axis_labels and col_id == 0:
            ax.set_ylabel('left rank index')
        else:
            ax.set_ylabel('')

    cbar = fig.colorbar(
        im,
        ax=axes[:n],
        fraction=0.025,
        pad=0.02,
        shrink=0.95,
    )
    if normalize == 'global':
        cbar.set_label('normalized absolute contribution (global 0–1)')
    else:
        cbar.set_label('normalized absolute contribution (per-core 0–1)')

    fig.suptitle(
        title,
        fontsize=14,
    )

    fig.set_constrained_layout_pads(
        w_pad=wspace,
        h_pad=hspace,
        hspace=hspace,
        wspace=wspace,
    )

    plt.show()


@torch.no_grad()
def compare_tt_forward_and_dense_reconstruction(model, tokenizer, module_names, prompt: str):
    encoded = tokenizer(prompt, return_tensors='pt')
    encoded = {k: v.to(infer_input_device(model)) for k, v in encoded.items()}
    tt_logits = model(**encoded).logits.detach().float().cpu()
    replace_tt_with_dense_reconstruction(model, module_names)
    dense_logits = model(**encoded).logits.detach().float().cpu()
    diff = (tt_logits - dense_logits).abs()
    return {
        'max_abs_logit_diff': float(diff.max().item()),
        'mean_abs_logit_diff': float(diff.mean().item()),
    }

In [ ]:
def get_transformer_layers(model):
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        return model.model.layers
    raise ValueError('Expected a Hugging Face-style model with model.layers')


@torch.no_grad()
def capture_transformer_layer_output_maxima(model, tokenizer, prompt: str):
    """
    Capture max-magnitude output activation of each transformer block.
    Returns a DataFrame with one row per layer.
    """
    layers = get_transformer_layers(model)
    rows = []
    handles = []

    def make_hook(layer_idx: int):
        def hook(module, inputs, outputs):
            tensor = outputs[0] if isinstance(outputs, tuple) else outputs
            tensor = tensor.detach()
            flat_idx = int(tensor.abs().reshape(-1).argmax().item())
            unr = torch.unravel_index(torch.tensor(flat_idx, device=tensor.device), tensor.shape)
            unr = tuple(int(x.item()) for x in unr)
            value = float(tensor[unr].float().cpu().item())
            row = {
                'layer_idx': layer_idx,
                'value': value,
                'abs_value': abs(value),
            }
            if len(unr) >= 1:
                row['batch_idx'] = unr[0]
            if len(unr) >= 2:
                row['token_idx'] = unr[1]
            if len(unr) >= 3:
                row['channel_idx'] = unr[2]
            rows.append(row)
        return hook

    for layer_idx, layer in enumerate(layers):
        handles.append(layer.register_forward_hook(make_hook(layer_idx)))

    encoded = tokenizer(prompt, return_tensors='pt')
    encoded = {k: v.to(infer_input_device(model)) for k, v in encoded.items()}
    model.eval()
    _ = model(**encoded)

    for h in handles:
        h.remove()

    return pd.DataFrame(sorted(rows, key=lambda x: x['layer_idx']))


def build_ablation_coordinate_tables(
    model,
    superweight_df: pd.DataFrame,
    *,
    topk_outliers: int,
    random_samples: int,
    seed: int = 0,
):
    """
    Build coordinate tables for:
      - superweight
      - top_outlier (excluding superweights)
      - random (excluding superweights and chosen top outliers)
    """
    rng = torch.Generator().manual_seed(seed)

    tables = {
        'superweight': [],
        'top_outlier': [],
        'random': [],
    }

    grouped = superweight_df.groupby(['module_name', 'layer_idx'], sort=True)

    for (module_name, layer_idx), group in grouped:
        full_module_name = weight_coordinate_to_module_name(module_name, int(layer_idx))
        module = get_module_by_name(model, full_module_name)
        weight = module.weight.detach().float().cpu()
        H, W = weight.shape

        sw_coords = {(int(r.row), int(r.col)) for r in group.itertuples(index=False)}
        for (row, col) in sw_coords:
            tables['superweight'].append({
                'module_name': module_name,
                'layer_idx': int(layer_idx),
                'row': int(row),
                'col': int(col),
            })

        flat_abs = weight.abs().reshape(-1)
        top_idx = torch.topk(flat_abs, k=min(flat_abs.numel(), topk_outliers + len(sw_coords))).indices.tolist()

        top_coords = []
        for idx in top_idx:
            row = idx // W
            col = idx % W
            if (row, col) in sw_coords:
                continue
            top_coords.append((row, col))
            if len(top_coords) >= topk_outliers:
                break

        for (row, col) in top_coords:
            tables['top_outlier'].append({
                'module_name': module_name,
                'layer_idx': int(layer_idx),
                'row': int(row),
                'col': int(col),
            })

        forbidden = sw_coords | set(top_coords)
        all_coords = [(r, c) for r in range(H) for c in range(W) if (r, c) not in forbidden]
        if len(all_coords) > 0:
            perm = torch.randperm(len(all_coords), generator=rng).tolist()
            chosen = [all_coords[i] for i in perm[:min(random_samples, len(all_coords))]]
        else:
            chosen = []

        for (row, col) in chosen:
            tables['random'].append({
                'module_name': module_name,
                'layer_idx': int(layer_idx),
                'row': int(row),
                'col': int(col),
            })

    return {k: pd.DataFrame(v) for k, v in tables.items()}


def zero_selected_weights_(model, coord_df: pd.DataFrame):
    """
    Zero selected dense weight entries in-place.
    """
    restore_payload = []
    if coord_df is None or len(coord_df) == 0:
        return restore_payload

    for row in coord_df.itertuples(index=False):
        full_module_name = weight_coordinate_to_module_name(row.module_name, int(row.layer_idx))
        module = get_module_by_name(model, full_module_name)
        rr = int(row.row)
        cc = int(row.col)
        old_val = module.weight.data[rr, cc].detach().clone()
        module.weight.data[rr, cc] = 0
        restore_payload.append((full_module_name, rr, cc, old_val))
    return restore_payload


def restore_selected_weights_(model, restore_payload):
    for full_module_name, rr, cc, old_val in restore_payload:
        module = get_module_by_name(model, full_module_name)
        module.weight.data[rr, cc] = old_val.to(device=module.weight.device, dtype=module.weight.dtype)


def plot_layer_output_curves(curves: dict, title: str):
    plt.figure(figsize=(9.5, 5.0))
    for label, df in curves.items():
        plt.plot(df['layer_idx'], df['value'], marker='o', label=label)
    plt.xlabel('Layer')
    plt.ylabel('Max layer output activation')
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Load baseline model


In [ ]:
model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
print(format_cuda_memory())

## Baseline superweight search

In [ ]:
input_profile = capture_layerwise_maxima(
    model,
    tokenizer,
    DISCOVERY_PROMPT,
    module_name='mlp.down_proj',
    capture='input',
)
output_profile = capture_layerwise_maxima(
    model,
    tokenizer,
    DISCOVERY_PROMPT,
    module_name='mlp.down_proj',
    capture='output',
)

input_df = pd.DataFrame([vars(x) for x in input_profile])
output_df = pd.DataFrame([vars(x) for x in output_profile])
plot_layerwise_maxima(input_df, output_df, f'{MODEL_NAME} down_proj maxima')

candidates = find_superweight_candidates(
    model,
    tokenizer,
    DISCOVERY_PROMPT,
    module_name='mlp.down_proj',
    min_abs_value=DISCOVERY_THRESHOLD,
    require_same_token=REQUIRE_SAME_TOKEN,
)

candidate_df = candidates_table(candidates)
display(candidate_df)

In [ ]:
if USE_KNOWN_SUPERWEIGHTS and MODEL_NAME in KNOWN_SUPERWEIGHTS:
    superweight_df = known_superweight_rows(MODEL_NAME)
else:
    discovered = find_superweights_iterative(
        model,
        tokenizer,
        DISCOVERY_PROMPT,
        module_name='mlp.down_proj',
        min_abs_value=DISCOVERY_THRESHOLD,
        require_same_token=REQUIRE_SAME_TOKEN,
        max_superweights=MAX_SUPERWEIGHTS,
        select='earliest',
    )
    superweight_df = pd.DataFrame([
        {
            'module_name': coord.module_name,
            'layer_idx': coord.layer_idx,
            'row': coord.row,
            'col': coord.col,
            'value': value,
        }
        for coord, value in discovered.items()
    ])

if 'value' not in superweight_df.columns:
    values = []
    for row in superweight_df.itertuples(index=False):
        module = get_module_by_name(model, weight_coordinate_to_module_name(row.module_name, row.layer_idx))
        values.append(float(module.weight[row.row, row.col].float().cpu().item()))
    superweight_df['value'] = values

display(superweight_df)

In [ ]:
ablation_coord_tables = build_ablation_coordinate_tables(
    model,
    superweight_df,
    topk_outliers=PLOT_TOPK_OUTLIERS,
    random_samples=PLOT_RANDOM_SAMPLES,
    seed=0,
)

for tag, df in ablation_coord_tables.items():
    print(tag, len(df))
    display(df.head())

## Dense baseline diagnostics for the target layers


In [ ]:
dense_stats = []
for layer_idx in sorted(superweight_df['layer_idx'].unique()):
    module = get_module_by_name(model, weight_coordinate_to_module_name('mlp.down_proj', int(layer_idx)))
    layer_coords = [
        (int(row.row), int(row.col))
        for row in superweight_df.itertuples(index=False)
        if int(row.layer_idx) == int(layer_idx)
    ]
    layer_df = dense_weight_rank_stats(module.weight.detach().float().cpu(), layer_coords)
    layer_df.insert(0, 'layer_idx', int(layer_idx))
    dense_stats.append(layer_df)

dense_stats_df = pd.concat(dense_stats, ignore_index=True) if dense_stats else pd.DataFrame()
display(dense_stats_df)

print('Baseline generation preview:')
print(generate_text(model, tokenizer, GENERATION_PROMPT, max_new_tokens=48))

## Baseline layer-output ablations

In [ ]:
baseline_layer_output_df = capture_transformer_layer_output_maxima(
    model,
    tokenizer,
    DISCOVERY_PROMPT,
)

baseline_ablation_curves = {
    'original': baseline_layer_output_df,
}

for tag in ['superweight', 'top_outlier', 'random']:
    restore_payload = zero_selected_weights_(model, ablation_coord_tables[tag])
    baseline_ablation_curves[f'zero {tag}'] = capture_transformer_layer_output_maxima(
        model,
        tokenizer,
        DISCOVERY_PROMPT,
    )
    restore_selected_weights_(model, restore_payload)

plot_layer_output_curves(
    baseline_ablation_curves,
    title=f'{MODEL_NAME}: dense model layer-output ablations',
)

## TT rank sweep

In [ ]:
def analyze_tt_rank(tt_rank: int):
    run_model, run_tokenizer = load_model_and_tokenizer(MODEL_NAME)

    target_layers = sorted(int(v) for v in superweight_df['layer_idx'].unique())
    original_weights = {}
    for layer_idx in target_layers:
        mod = get_module_by_name(run_model, weight_coordinate_to_module_name('mlp.down_proj', layer_idx))
        original_weights[layer_idx] = mod.weight.detach().float().cpu().contiguous()

    summaries = replace_llama_ffn_with_tt(
        run_model,
        layer_indices=target_layers,
        tt_rank=tt_rank,
        order=ORDER,
        projections=PROJECTIONS,
        decompose_dtype=DECOMPOSE_DTYPE,
        decompose_device=DECOMPOSE_DEVICE,
        token_chunk_size=TOKEN_CHUNK_SIZE,
    )

    tt_module_names = [s.module_name for s in summaries]

    tt_layer_output_df = capture_transformer_layer_output_maxima(
        run_model,
        run_tokenizer,
        DISCOVERY_PROMPT,
    )

    entry_rows = []
    trace_tables = {}
    error_cloud = []
    contribution_payloads = {}

    for row in superweight_df.itertuples(index=False):
        layer_idx = int(row.layer_idx)
        module_name = weight_coordinate_to_module_name(row.module_name, layer_idx)
        tt_module = get_module_by_name(run_model, module_name)
        entry_error = compare_entry_with_dense(tt_module, original_weights[layer_idx], int(row.row), int(row.col))
        entry_rows.append({
            'tt_rank': tt_rank,
            'layer_idx': layer_idx,
            'row': int(row.row),
            'col': int(row.col),
            'original_value': entry_error.original_value,
            'tt_value': entry_error.tt_value,
            'abs_error': entry_error.abs_error,
            'rel_error': entry_error.rel_error,
            'compression_ratio': float(tt_module.compression_ratio()),
        })
        trace_tables[(layer_idx, int(row.row), int(row.col))] = entry_trace_table(tt_module, int(row.row), int(row.col))
        contribution_payloads[(layer_idx, int(row.row), int(row.col))] = tt_module

        error_cloud.append(
            sample_entry_error_summary(
                tt_module,
                original_weights[layer_idx],
                superweight_coords=[(int(row.row), int(row.col))],
                topk_outliers=PLOT_TOPK_OUTLIERS,
                random_samples=PLOT_RANDOM_SAMPLES,
                seed=tt_rank,
            ).assign(layer_idx=layer_idx)
        )

    tt_output_profile = capture_layerwise_maxima(
        run_model,
        run_tokenizer,
        ANALYSIS_PROMPT,
        module_name='mlp.down_proj',
        capture='output',
    )
    tt_output_df = pd.DataFrame([vars(x) for x in tt_output_profile])

    tt_vs_dense = compare_tt_forward_and_dense_reconstruction(
        run_model,
        run_tokenizer,
        tt_module_names,
        prompt=GENERATION_PROMPT,
    )

    dense_recon_layer_output_df = capture_transformer_layer_output_maxima(
        run_model,
        run_tokenizer,
        DISCOVERY_PROMPT,
    )

    dense_recon_ablation_curves = {
        'original_reconstructed_dense': dense_recon_layer_output_df,
    }

    for tag in ['superweight', 'top_outlier', 'random']:
        restore_payload = zero_selected_weights_(run_model, ablation_coord_tables[tag])
        dense_recon_ablation_curves[f'zero {tag}'] = capture_transformer_layer_output_maxima(
            run_model,
            run_tokenizer,
            DISCOVERY_PROMPT,
        )
        restore_selected_weights_(run_model, restore_payload)

    dense_recon_output_profile = capture_layerwise_maxima(
        run_model,
        run_tokenizer,
        ANALYSIS_PROMPT,
        module_name='mlp.down_proj',
        capture='output',
    )
    dense_recon_output_df = pd.DataFrame([vars(x) for x in dense_recon_output_profile])

    ppl_tt = None
    ppl_dense = None
    if RUN_PPL and eval_ppl is not None:
        run_model_tt, run_tokenizer_tt = load_model_and_tokenizer(MODEL_NAME)
        replace_llama_ffn_with_tt(
            run_model_tt,
            layer_indices=target_layers,
            tt_rank=tt_rank,
            order=ORDER,
            projections=PROJECTIONS,
            decompose_dtype=DECOMPOSE_DTYPE,
            decompose_device=DECOMPOSE_DEVICE,
            token_chunk_size=TOKEN_CHUNK_SIZE,
        )
        ppl_tt = eval_ppl(run_model_tt, run_tokenizer_tt, datasets=DATASETS, seqlen=SEQLEN)
        replace_tt_with_dense_reconstruction(
            run_model_tt,
            [f'model.layers.{layer_idx}.mlp.down_proj' for layer_idx in target_layers],
        )
        ppl_dense = eval_ppl(run_model_tt, run_tokenizer_tt, datasets=DATASETS, seqlen=SEQLEN)
        clean_memory(run_model_tt, run_tokenizer_tt)

    result = {
        'tt_rank': tt_rank,
        'entry_errors': pd.DataFrame(entry_rows),
        'trace_tables': trace_tables,
        'error_cloud': pd.concat(error_cloud, ignore_index=True),
        'tt_output_df': tt_output_df,
        'dense_recon_output_df': dense_recon_output_df,
        'tt_vs_dense': tt_vs_dense,
        'ppl_tt': ppl_tt,
        'ppl_dense_reconstruction': ppl_dense,
        'tt_module_names': tt_module_names,
        'contribution_payloads': contribution_payloads,
        'tt_layer_output_df': tt_layer_output_df,
        'dense_recon_layer_output_df': dense_recon_layer_output_df,
        'dense_recon_ablation_curves': dense_recon_ablation_curves,
    }

    clean_memory(run_model, run_tokenizer)
    return result

In [ ]:
rank_results = []
for tt_rank in TT_RANKS:
    print(f'Running TT rank = {tt_rank}')
    rank_results.append(analyze_tt_rank(tt_rank))
    print('done')

In [ ]:
summary_rows = []
for result in rank_results:
    entry_df = result['entry_errors']
    cloud = result['error_cloud']

    def mean_for(tag, col):
        sub = cloud[cloud['tag'] == tag]
        return float(sub[col].mean()) if len(sub) > 0 else float('nan')

    def max_for(tag, col):
        sub = cloud[cloud['tag'] == tag]
        return float(sub[col].max()) if len(sub) > 0 else float('nan')
    
    def mean_for_filtered(tag, col, min_original_abs=None):
        sub = cloud[cloud['tag'] == tag]
        if min_original_abs is not None:
            sub = sub[sub['original_abs'] >= min_original_abs]
        return float(sub[col].mean()) if len(sub) > 0 else float('nan')
    
    def max_for_filtered(tag, col, min_original_abs=None):
        sub = cloud[cloud['tag'] == tag]
        if min_original_abs is not None:
            sub = sub[sub['original_abs'] >= min_original_abs]
        return float(sub[col].max()) if len(sub) > 0 else float('nan')

    row = {
        'tt_rank': result['tt_rank'],

        # superweights
        'mean_superweight_rel_error': float(entry_df['rel_error'].mean()),
        'max_superweight_rel_error': float(entry_df['rel_error'].max()),
        'mean_superweight_abs_error': float(entry_df['abs_error'].mean()),

        'mean_top_outlier_rel_error': mean_for('top_outlier', 'rel_error'),
        'max_top_outlier_rel_error': max_for('top_outlier', 'rel_error'),
        'mean_top_outlier_abs_error': mean_for('top_outlier', 'abs_error'),

        'mean_random_abs_error': mean_for('random', 'abs_error'),
        'max_random_abs_error': max_for('random', 'abs_error'),
        'mean_random_rel_error': mean_for_filtered('random', 'rel_error', min_original_abs=1e-3),
        'max_random_rel_error': max_for_filtered('random', 'rel_error', min_original_abs=1e-3),

        'superweight_vs_top_outlier_rel_error': (
            float(entry_df['rel_error'].mean()) / max(mean_for('top_outlier', 'rel_error'), 1e-12)
            if not math.isnan(mean_for('top_outlier', 'rel_error')) else float('nan')
        ),

        'max_abs_logit_diff_tt_vs_dense_reconstruction': result['tt_vs_dense']['max_abs_logit_diff'],
        'mean_abs_logit_diff_tt_vs_dense_reconstruction': result['tt_vs_dense']['mean_abs_logit_diff'],
    }

    if result['ppl_tt'] is not None:
        for ds, value in result['ppl_tt'].items():
            row[f'ppl_tt_{ds}'] = value
    if result['ppl_dense_reconstruction'] is not None:
        for ds, value in result['ppl_dense_reconstruction'].items():
            row[f'ppl_dense_reconstruction_{ds}'] = value

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
summary_df.to_json(OUTPUT_JSON, orient='records', indent=2)
print('Saved summary to', OUTPUT_JSON)

## How TT affects the superweights

In [ ]:
plt.figure(figsize=(7, 4.8))
plt.plot(
    summary_df['tt_rank'],
    summary_df['mean_superweight_rel_error'],
    marker='o',
    label='superweights',
)
plt.plot(
    summary_df['tt_rank'],
    summary_df['mean_top_outlier_rel_error'],
    marker='s',
    label='other top outliers',
)
plt.plot(summary_df['tt_rank'], summary_df['mean_random_rel_error'], marker='^', label='random (|w| >= 1e-3)')

#plt.xscale('log')
#plt.yscale('log')
plt.xlabel('TT rank')
plt.ylabel('Mean relative entry error')
plt.title('Reconstruction relative error after TT')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4.8))
plt.plot(
    summary_df['tt_rank'],
    summary_df['mean_superweight_abs_error'],
    marker='o',
    label='superweights',
)
plt.plot(
    summary_df['tt_rank'],
    summary_df['mean_top_outlier_abs_error'],
    marker='s',
    label='other top outliers',
)
plt.plot(summary_df['tt_rank'], summary_df['mean_random_abs_error'], marker='^', label='random')

#plt.xscale('log')
#plt.yscale('log')
plt.xlabel('TT rank')
plt.ylabel('Mean absolute entry error')
plt.title('Reconstruction absolute error after TT')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
for result in rank_results:
    print(f"\nTT rank = {result['tt_rank']}")
    display(result['entry_errors'])

## Compare TT-forward activation spikes

In [ ]:
baseline_output_df = output_df[['layer_idx', 'value']].rename(columns={'value': 'baseline'})

for result in rank_results:
    merged = baseline_output_df.merge(
        result['tt_output_df'][['layer_idx', 'value']].rename(columns={'value': 'tt_forward'}),
        on='layer_idx',
        how='left',
    ).merge(
        result['dense_recon_output_df'][['layer_idx', 'value']].rename(columns={'value': 'dense_reconstruction'}),
        on='layer_idx',
        how='left',
    )

    plt.figure(figsize=(7, 4.5))
    plt.plot(merged['layer_idx'], merged['baseline'], marker='o', label='baseline')
    plt.plot(merged['layer_idx'], merged['tt_forward'], marker='s', label='TT forward')
    plt.xlabel('Layer')
    plt.ylabel('Max down_proj output activation')
    plt.title(f'Activation spike retention, TT rank = {result["tt_rank"]}')
    plt.grid(True)
    plt.legend()
    plt.show()

## Layer-output maxima: dense baseline vs TT-forward

In [ ]:
for result in rank_results:
    plot_layer_output_curves(
        {
            'dense baseline': baseline_layer_output_df,
            f'TT forward (rank={result["tt_rank"]})': result['tt_layer_output_df'],
        },
        title=f'{MODEL_NAME}: dense baseline vs TT-forward (rank={result["tt_rank"]})',
    )

## Layer-output ablations: dense baseline and reconstructed dense models

In [ ]:
plot_layer_output_curves(
    baseline_ablation_curves,
    title=f'{MODEL_NAME}: dense baseline ablations',
)

In [ ]:
for result in rank_results:
    plot_layer_output_curves(
        result['dense_recon_ablation_curves'],
        title=f'{MODEL_NAME}: reconstructed dense ablations (from TT rank={result["tt_rank"]})',
    )

## Compare superweights against ordinary outliers

In [ ]:
for result in rank_results:
    cloud = result['error_cloud']
    plot_error_scatter(cloud, f'Dense vs TT entry errors (rank={result["tt_rank"]})', use_log=True)
    display(cloud.groupby('tag')[['abs_error', 'rel_error']].agg(['mean', 'max', 'median']))

In [ ]:
for result in rank_results:
    cloud = result['error_cloud']
    plot_error_scatter(cloud, f'Dense vs TT entry errors (rank={result["tt_rank"]})', use_log=False)
    display(cloud.groupby('tag')[['abs_error', 'rel_error']].agg(['mean', 'max', 'median']))

## How one dense superweight is distributed across TT cores

In [ ]:
HEATMAP_ZERO_COLOR = 'black'
HEATMAP_FIGSIZE_PER_CORE = 3
HEATMAP_PANEL_SCALE = 1.15

for select_rank in TT_RANKS:
    selected = [result for result in rank_results if result['tt_rank'] == select_rank][0]
    selected_coord = superweight_df.iloc[0]
    
    layer_idx = int(selected_coord.layer_idx)
    row = int(selected_coord.row)
    col = int(selected_coord.col)
    
    print(f'Showing TT-core trace for rank={select_rank}, layer={layer_idx}, coord=({row}, {col})')
    
    selected_module = selected['contribution_payloads'][(layer_idx, row, col)]
    
    plot_core_contribution_maps(
        selected_module,
        title=f'Per-core contribution maps for entry ({row}, {col}) for rank {select_rank}',
        row=row,
        col=col,
        figsize_per_core=HEATMAP_FIGSIZE_PER_CORE,
        panel_scale=HEATMAP_PANEL_SCALE,
        cmap_name='plasma',
        zero_color=HEATMAP_ZERO_COLOR,
        zero_tol=1e-2,
        normalize='global',
        use_abs=True,
        wspace=0.05,
        hspace=0.05
    )